# Complementing Regular Expressions

In the lecture notes we have shown that the complement $\Sigma^* \backslash L$ of a regular language $L$ over the
alphabet $\Sigma$ is again a regular language.  The proof converts a regular expression $r$ with $L = L(r)$ into an
<span style="font-variant:small-caps;">Nfa</span> $F$, which is then converted into the complete <span style="font-variant:small-caps;">Dfa</span> $\mathit{det}(F)$.  Swapping the accepting and the
non-accepting states of this <span style="font-variant:small-caps;">Dfa</span> yields a <span style="font-variant:small-caps;">Dfa</span> that accepts the complement $\Sigma^* \backslash L(r)$.
Therefore, there is a function
$$ \texttt{complementRegExp}: \textsf{RegExp} \rightarrow \textsf{RegExp} $$
such that
$$ L\bigl(\texttt{complementRegExp}(r)\bigr) = \Sigma^* \backslash L(r) $$
holds for every regular expression $r$.  In this notebook we implement this function.  As the complement is taken
with respect to $\Sigma^*$, the function also needs the alphabet $\Sigma$ as an argument:  For example, the
complement of $L(\texttt{a}^*)$ is the empty language if $\Sigma = \{\texttt{a}\}$, but it is the set of all
strings that contain a $\texttt{b}$ if $\Sigma = \{\texttt{a}, \texttt{b}\}$.  Given a regular expression $r$
and an alphabet $\Sigma$, a regular expression for the complement of $L(r)$ is computed as follows:
1. $r$ is converted into an <span style="font-variant:small-caps;">Nfa</span>, which is then converted into a complete <span style="font-variant:small-caps;">Dfa</span> $F$.
2. The <span style="font-variant:small-caps;">Dfa</span> $F$ is minimized.  This step is not needed for the construction, but it keeps the <span style="font-variant:small-caps;">Dfa</span> small.
3. We compute the complement automaton $\overline{F}$ by swapping the accepting and the non-accepting states of $F$.
4. The complement automaton is converted into a regular expression.

We will see that the resulting regular expression is much larger than necessary, even for very simple regular
expressions $r$.

The notebook `09-Equivalence.ipynb` provides the function `regexp2DFA`, which converts a regular expression into a <span style="font-variant:small-caps;">Dfa</span>, and the functions `fsm_complement` and `is_empty`, which we use to check our results.

In [ ]:
#use "../lib/nb.ml";;
run_notebook "09-Equivalence.ipynb"

The notebook `07-Minimize.ipynb` provides the function `minimize`, which minimizes a <span style="font-variant:small-caps;">Dfa</span>.

In [ ]:
run_notebook "07-Minimize.ipynb"

The notebook `05-DFA-2-RegExp.ipynb` provides the function `dfa_2_regexp`, which converts a <span style="font-variant:small-caps;">Dfa</span> into a regular expression.

In [ ]:
run_notebook "05-DFA-2-RegExp.ipynb"

The notebook `Simplify.ipynb` provides the function `simplify`, which simplifies a regular expression, and the function `to_str`, which turns a regular expression into a string that contains only the parentheses that are needed.  The notebook `RegExp-Parser.ipynb` provides the function `parse`, which turns a string into a regular expression.

In [ ]:
run_notebook "Simplify.ipynb";;
run_notebook "RegExp-Parser.ipynb"

**Note:** The function `run_notebook` executes all cells of a notebook, just like the magic command `%run`.  The
Python version has to mark the calls of functions that are defined in the loaded notebooks with
`# type: ignore`.  The OCaml compiler knows these functions, so every call is type checked.

The regular expressions computed in this notebook are huge.  Therefore, the directive `#print_depth 4` tells the
notebook to show only the top four levels of nested values.  The function `Unix.gettimeofday`, which is used to
measure the running time, corresponds to Python's `time.perf_counter`.

In [ ]:
#print_depth 4

In [ ]:
#show_type regexp;;
#show_type dfa

Given a complete <span style="font-variant:small-caps;">Dfa</span>
$$ F = \langle Q, \Sigma, \delta, q_0, A \rangle, $$
the *complement automaton* $\overline{F}$ is defined as
$$ \overline{F} := \langle Q, \Sigma, \delta, q_0, Q \backslash A \rangle, $$
i.e. the accepting states of $\overline{F}$ are the states of $F$ that are not accepting.  As $F$ is complete,
reading a string $w$ takes $F$ into the state $\delta^*(q_0, w)$, and we have
$$ w \in L\bigl(\overline{F}\bigr) \;\Leftrightarrow\; \delta^*(q_0, w) \in Q \backslash A
   \;\Leftrightarrow\; w \not\in L(F). $$
Hence $\overline{F}$ accepts the language $\Sigma^* \backslash L(F)$.  The construction requires that $F$ is
complete:  If $F$ got stuck on a string $w$ because $\delta$ is undefined for some state and character, then
neither $F$ nor $\overline{F}$ would accept $w$.  The <span style="font-variant:small-caps;">Dfa</span>s computed by the function `nfa2dfa` are complete:
If a character leads to no state of the <span style="font-variant:small-caps;">Nfa</span>, the corresponding transition leads to the empty set, which is then a
state of the <span style="font-variant:small-caps;">Dfa</span>.  The function `minimize` keeps the <span style="font-variant:small-caps;">Dfa</span>s complete.

**Function `complement_automaton f`**
- *Input:* `f` is a complete <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* The complement automaton $\overline{F}$, which accepts the language $\Sigma^* \backslash L(F)$.

In [ ]:
let complement_automaton (f : 'q dfa) : 'q dfa =
  let ({ states; sigma; delta; start = q0; accepting = a } : 'q dfa) = f in
  { states; sigma; delta; start = q0; accepting = FSet.diff states a }

**Function `complement_dfa r sigma`**
- *Input:* `r` is a regular expression and `sigma` is the alphabet.
- *Output:* The complement automaton of the minimized <span style="font-variant:small-caps;">Dfa</span> for `r`.  It accepts the language
  $\Sigma^* \backslash L(r)$.

In [ ]:
let complement_dfa (r : regexp) (sigma : char FSet.t) : state FSet.t min_dfa =
  let f = minimize (regexp2DFA r sigma) in
  complement_automaton f

**Function `complementRegExp r sigma`**
- *Input:* `r` is a regular expression and `sigma` is the alphabet.
- *Output:* A regular expression that describes the language $\Sigma^* \backslash L(r)$.  It is not simplified.

In [ ]:
let complementRegExp (r : regexp) (sigma : char FSet.t) : regexp =
  dfa_2_regexp (complement_dfa r sigma)

To measure the size of a regular expression we count the nodes of its syntax tree:  Every character, every
occurrence of $\varepsilon$ and $\emptyset$, and every operator counts as one node.

**Function `size r`**
- *Input:* `r` is a regular expression, represented as a value of type `regexp`.
- *Output:* The number of nodes of the syntax tree of `r`.

In [ ]:
let rec size (r : regexp) : int =
  match r with
  | Star r1 -> 1 + size r1
  | Sum (r1, r2) | Product (r1, r2) -> 1 + size r1 + size r2
  | Empty | Epsilon | Char _ -> 1

The function `accepts_same_language` checks whether the <span style="font-variant:small-caps;">Dfa</span> computed from a regular expression `r` accepts
the same language as the <span style="font-variant:small-caps;">Dfa</span> `f`.  We use it to check that the regular expressions computed by
`complementRegExp` are correct.

**Function `accepts_same_language r f sigma`**
- *Input:* `r` is a regular expression, `f` is a complete <span style="font-variant:small-caps;">Dfa</span>, and `sigma` is its alphabet.
- *Output:* `true` if $L(r) = L(F)$, `false` otherwise.

In [ ]:
let accepts_same_language (r : regexp) (f : 'q dfa) (sigma : char FSet.t) : bool =
  let d = regexp2DFA r sigma in
  is_empty (fsm_complement d f) && is_empty (fsm_complement f d)

## A First Example

The regular expression $r$ describes the strings over the alphabet $\{\texttt{a}, \texttt{b}\}$ that contain the
substring $\texttt{ab}$:
$$ r = (\texttt{a} + \texttt{b})^* \cdot \texttt{a} \cdot \texttt{b} \cdot (\texttt{a} + \texttt{b})^*. $$

In [ ]:
let sigma = FSet.of_list [ 'a'; 'b' ]
let r     = parse "(a+b)*ab(a+b)*"
;;
size r

The minimized <span style="font-variant:small-caps;">Dfa</span> for $r$ has 3 states and so has its complement automaton.

In [ ]:
let f = complement_dfa r sigma
;;
FSet.cardinal f.states

In [ ]:
let c = complementRegExp r sigma
;;
size c

Although $r$ has only 13 nodes, the regular expression for its complement has 425 nodes.
Let us check that it is correct.

**Note:** The Python version of this notebook computes a regular expression with 415 nodes.  The function
`rpq` eliminates the states of the automaton one after the other, and the size of the result depends on the
order in which the states are eliminated.  The elements of an OCaml set of type `FSet.t` are sorted, while Python
stores the elements of a set in an order that depends on their hash values.  For the same reason, some of the
sizes in the table below differ from those of the Python version.  Of course, all of the results are correct.

In [ ]:
accepts_same_language c f sigma

After simplification, the regular expression is much smaller:

In [ ]:
let s = simplify c
;;
size s

In [ ]:
print_endline (to_str s)

In [ ]:
accepts_same_language s f sigma

This is exactly the regular expression that a human would write:  A string does not contain the substring
$\texttt{ab}$ iff no $\texttt{b}$ follows an $\texttt{a}$, i.e. iff all of its $\texttt{b}$s precede all of its
$\texttt{a}$s.  This leads to the regular expression
$$ \texttt{b}^* \cdot \texttt{a}^*. $$

In [ ]:
s = parse "b*a*"

## Growth of the Regular Expressions

The function `dfa_2_regexp` calls the function `rpq`, which calls itself four times for every state of the
<span style="font-variant:small-caps;">Dfa</span>.  Therefore, the size of the regular expression computed for a <span style="font-variant:small-caps;">Dfa</span> with $n$ states grows like $4^n$.
As the complement automaton has as many states as the minimized <span style="font-variant:small-caps;">Dfa</span> for $r$, the size of the complement grows
exponentially in this number.  The following table shows this for the languages of strings in which the number of
`a`s is divisible by $k$.  The complements of these languages consist of the strings in which the number of `a`s is
not divisible by $k$.  For $k = 1$, the language contains all strings, so its complement is empty.

**Function `divisible c d k`**
- *Input:* `c` and `d` are the two characters of the alphabet and `k` is a positive natural number.
- *Output:* A string that is parsed into a regular expression describing the strings over the alphabet
  $\{c, d\}$ in which the number of occurrences of `c` is divisible by `k`.

In [ ]:
let divisible (c : char) (d : char) (k : int) : string =
  "(" ^ String.concat "" (List.init k (fun _ -> Printf.sprintf "%c*%c" d c)) ^ Printf.sprintf ")*%c*" d

In [ ]:
print_endline (divisible 'a' 'b' 2);
print_endline (divisible 'a' 'b' 3)

In [ ]:
print_endline
  (String.concat " "
     [ rjust "k" 2; rjust "states" 7; rjust "size(r)" 8; rjust "size(complement)" 17; rjust "simplified" 11;
       rjust "correct" 8; rjust "time" 7 ]);
for k = 1 to 8 do
  let r     = parse (divisible 'a' 'b' k) in
  let start = Unix.gettimeofday () in
  let f     = complement_dfa r sigma in
  let c     = dfa_2_regexp f in
  let s     = simplify c in
  let stop  = Unix.gettimeofday () in
  let ok    = accepts_same_language s f sigma in
  Printf.printf "%2d %7d %8d %17d %11d %8b %6.2fs\n%!" k (FSet.cardinal f.states) (size r) (size c) (size s) ok (stop -. start)
done

Although the regular expressions $r$ are tiny, the regular expression for the complement has almost
1&nbsp;400&nbsp;000 nodes when the minimized <span style="font-variant:small-caps;">Dfa</span> has only eight states.  Simplification removes almost all of
these nodes:  For $k = 8$, the simplified regular expression has only 86 nodes.  Nevertheless, we do not compute
the complement of a regular expression in practice, since the regular expression computed by `dfa_2_regexp` grows
exponentially:  If we need the complement of a regular language, we work with the complement automaton instead.